In [68]:
# Installing the core framework, AI connectors, and reporting tools
!pip install -qU langgraph langchain_openai langchain_community tavily-python
!pip install -qU langchain-tavily
!pip install -qU langchain-groq
!pip install -qU fpdf2 markdown

In [69]:
# Initializing Groq and Tavily
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from tavily import TavilyClient

llm = ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    groq_api_key=userdata.get('GROQ_API_KEY')
)

search_tool = TavilyClient(api_key=userdata.get('TAVILY_API_KEY'))

print("Setup Complete: LLM and Search Tool are connected.")

Setup Complete: LLM and Search Tool are connected.


In [70]:
# Defining the Shared State that all agents use to exchange information
from typing import Annotated, List, TypedDict

class ResearchState(TypedDict):
    task: str
    queries: List[str]
    sources: List[dict]
    evidence_clusters: dict
    contradictions: List[str]
    report: str
    log: List[str]

In [71]:
# Agent 1: The Search Agent - Expands the task into queries and fetches live web data
import json
import re

def search_agent(state: ResearchState):
    print("--- SEARCHING FOR SOURCES ---")
    query_prompt = f"You are a Lead Researcher. Given the topic: {state['task']}, generate 3 distinct search queries. Output ONLY a JSON list of strings."
    response = llm.invoke(query_prompt)

    try:
        json_match = re.search(r'\[.*\]', response.content, re.DOTALL)
        queries = json.loads(json_match.group(0)) if json_match else []
    except:
        queries = [state['task']]

    all_results = []
    for q in queries:
        results = search_tool.search(query=q)['results']
        all_results.extend(results)

    return {"queries": queries, "sources": all_results, "log": state.get("log", []) + [f"Found {len(all_results)} sources."]}

In [72]:
# Agent 2: The Synthesis Agent - Groups search results into themes and identifies conflicting information
def synthesis_agent(state: ResearchState):
    print("--- SYNTHESIZING EVIDENCE ---")
    context = "\n".join([f"Source: {s.get('url')}\nContent: {s.get('content')}" for s in state['sources']])

    synthesis_prompt = f"""Analyze these regarding '{state['task']}':
    1. Group into themes. 2. Identify DISAGREEMENTS.
    Output ONLY valid JSON with keys: 'themes' (dict) and 'contradictions' (list).
    Context: {context}"""

    response = llm.invoke(synthesis_prompt)

    try:
        json_match = re.search(r'\{.*\}', response.content, re.DOTALL)
        analysis = json.loads(json_match.group(0)) if json_match else {"themes": {}, "contradictions": []}
    except Exception as e:
        analysis = {"themes": {"Summary": "Error parsing JSON, provided raw text summary."}, "contradictions": []}

    return {
        "evidence_clusters": analysis.get('themes', {}),
        "contradictions": analysis.get('contradictions', []),
        "log": state.get("log", []) + ["Synthesized evidence into clusters."]
    }

In [73]:
# Agent 3: The Report Agent - Transforms synthesized evidence and source URLs into a structured Markdown report
def report_agent(state: ResearchState):
    print("--- GENERATING FINAL RESEARCH REPORT ---")

    themes_data = json.dumps(state['evidence_clusters'], indent=2)
    contradictions_data = "\n".join(state['contradictions'])
    source_list = "\n".join([f"- {s.get('url')}" for s in state['sources']])

    report_prompt = f"""You are a Professional Research Writer. Create a comprehensive report for: {state['task']}

    THEMES: {themes_data}
    CONTRADICTIONS: {contradictions_data}
    SOURCES: {source_list}

    STRICT STRUCTURE:
    1. Title 2. Introduction 3. Thematic Findings (with [Source URL] citations)
    4. Discussion of Agreements/Disagreements 5. Limitations 6. Conclusion 7. References

    Format the entire response in clean, professional Markdown."""

    response = llm.invoke(report_prompt)

    return {
        "report": response.content,
        "log": state.get("log", []) + ["Generated full structured report with citations."]
    }

In [74]:
# Graph Construction: Assembling the agents into a linear workflow using LangGraph
from langgraph.graph import StateGraph, END

workflow = StateGraph(ResearchState)

# 1. Adding Nodes (The Workers)
workflow.add_node("searcher", search_agent)
workflow.add_node("synthesizer", synthesis_agent)
workflow.add_node("writer", report_agent)

# 2. Defining the Workflow (The Path)
workflow.set_entry_point("searcher")
workflow.add_edge("searcher", "synthesizer")
workflow.add_edge("synthesizer", "writer")
workflow.add_edge("writer", END)

# 3. Compiling the Application
app = workflow.compile()
print("Graph Compiled! The Multi-Agent System is fully assembled.")

Graph Compiled! The Multi-Agent System is fully assembled.


In [75]:
# Execution: Runing the multi-agent system on a research topic and display the Markdown result
from IPython.display import Markdown

# 1. Defining the research task
inputs = {"task": "What are the main trade-offs between CNNs and Vision Transformers for medical imaging?"}

# 2. Invoking the compiled graph
final_state = app.invoke(inputs)

# 3. Rendering the generated report in the notebook
Markdown(final_state['report'])

--- SEARCHING FOR SOURCES ---
--- SYNTHESIZING EVIDENCE ---
--- GENERATING FINAL RESEARCH REPORT ---


# Trade-Offs between CNNs and Vision Transformers for Medical Imaging
## 1. Title
Trade-Offs between CNNs and Vision Transformers for Medical Imaging: A Comprehensive Review

## 2. Introduction
The field of medical imaging has witnessed significant advancements with the introduction of deep learning techniques, particularly Convolutional Neural Networks (CNNs) and Vision Transformers (ViTs). Both architectures have been extensively used for various medical imaging tasks, including image classification, segmentation, and detection. However, the choice between CNNs and ViTs depends on several factors, including performance, computational efficiency, and application areas. This report aims to provide a comprehensive review of the trade-offs between CNNs and ViTs for medical imaging, highlighting their strengths, weaknesses, and potential applications.

## 3. Thematic Findings
The following thematic findings summarize the current state of research on CNNs and ViTs for medical imaging:

* **Performance Comparison**: ViTs have shown superior performance in some medical imaging tasks, such as detecting anomalies in radiological scans [https://openreview.net/forum?id=3Wybo29gGlx](https://openreview.net/forum?id=3Wybo29gGlx). However, CNNs have also demonstrated excellent performance in tasks that require high-resolution image analysis [https://pubmed.ncbi.nlm.nih.gov/39264388/](https://pubmed.ncbi.nlm.nih.gov/39264388/).
* **Application Areas**: ViTs have been found to be particularly useful in tasks that require the analysis of large-scale patterns and contextual information in images, such as identifying patterns on histopathology slides [https://www.reddit.com/r/computervision/comments/1cu3pnw/cnn_vs_vision_transformer_a_practitioners_guide/](https://www.reddit.com/r/computervision/comments/1cu3pnw/cnn_vs_vision_transformer_a_practitioners_guide/).
* **Hybrid Models**: Combining the strengths of both architectures, hybrid models that merge convolutional layers with transformers have been proposed, showing promising results in medical image classification tasks [https://pmc.ncbi.nlm.nih.gov/articles/PMC11393140/](https://pmc.ncbi.nlm.nih.gov/articles/PMC11393140/).
* **Computational Efficiency**: CNNs are generally more computationally efficient than ViTs, but the latter can be optimized for better performance [https://blog.roboflow.com/vision-transformer-vs-cnn-for-detection/](https://blog.roboflow.com/vision-transformer-vs-cnn-for-detection/). Some studies have shown that ViTs can scale better for cloud services [https://medium.com/@hassaanidrees7/vision-transformer-vs-cnn-a-comparison-of-two-image-processing-giants-d6c85296f34f](https://medium.com/@hassaanidrees7/vision-transformer-vs-cnn-a-comparison-of-two-image-processing-giants-d6c85296f34f).
* **Medical Imaging Tasks**: Both ViTs and CNNs have been applied to various medical imaging tasks, including image classification, segmentation, and detection, with varying degrees of success [https://starfishmedical.com/resource/computer-vision-attention-or-convolution/](https://starfishmedical.com/resource/computer-vision-attention-or-convolution/).

## 4. Discussion of Agreements/Disagreements
While some studies suggest that ViTs outperform CNNs in medical imaging tasks, others find that CNNs perform better, particularly in tasks that require high-resolution image analysis. There is also disagreement on the computational efficiency of ViTs compared to CNNs, with some studies finding that ViTs require more resources, while others claim that they can be optimized for better performance. The effectiveness of hybrid models that combine ViTs and CNNs is still a topic of debate, with some studies showing promising results, while others find that they do not significantly improve performance.

## 5. Limitations
This report has several limitations. Firstly, the review is based on a limited number of studies, and a more comprehensive review of the literature may reveal additional findings. Secondly, the report focuses primarily on the trade-offs between CNNs and ViTs, without considering other deep learning architectures that may be relevant for medical imaging tasks. Finally, the report does not provide a detailed analysis of the technical implementation of CNNs and ViTs, which may be important for practitioners working in the field.

## 6. Conclusion
In conclusion, the choice between CNNs and ViTs for medical imaging tasks depends on several factors, including performance, computational efficiency, and application areas. While ViTs have shown superior performance in some tasks, CNNs have also demonstrated excellent performance in tasks that require high-resolution image analysis. Hybrid models that combine the strengths of both architectures may offer a promising solution for medical image classification tasks. Further research is needed to fully explore the trade-offs between CNNs and ViTs and to develop more effective deep learning architectures for medical imaging tasks.

## 7. References
* [https://openreview.net/forum?id=3Wybo29gGlx](https://openreview.net/forum?id=3Wybo29gGlx)
* [https://pubmed.ncbi.nlm.nih.gov/39264388/](https://pubmed.ncbi.nlm.nih.gov/39264388/)
* [https://www.reddit.com/r/computervision/comments/1cu3pnw/cnn_vs_vision_transformer_a_practitioners_guide/](https://www.reddit.com/r/computervision/comments/1cu3pnw/cnn_vs_vision_transformer_a_practitioners_guide/)
* [https://pmc.ncbi.nlm.nih.gov/articles/PMC11393140/](https://pmc.ncbi.nlm.nih.gov/articles/PMC11393140/)
* [https://blog.roboflow.com/vision-transformer-vs-cnn-for-detection/](https://blog.roboflow.com/vision-transformer-vs-cnn-for-detection/)
* [https://medium.com/@hassaanidrees7/vision-transformer-vs-cnn-a-comparison-of-two-image-processing-giants-d6c85296f34f](https://medium.com/@hassaanidrees7/vision-transformer-vs-cnn-a-comparison-of-two-image-processing-giants-d6c85296f34f)
* [https://starfishmedical.com/resource/computer-vision-attention-or-convolution/](https://starfishmedical.com/resource/computer-vision-attention-or-convolution/)
* [https://ieeexplore.ieee.org/iel8/6287639/10820123/10962160.pdf](https://ieeexplore.ieee.org/iel8/6287639/10820123/10962160.pdf)
* [https://pmc.ncbi.nlm.nih.gov/articles/PMC11393140/](https://pmc.ncbi.nlm.nih.gov/articles/PMC11393140/)
* [https://www.researchgate.net/publication/394100604_Comparative_Analysis_of_Vision_Transformers_and_Convolutional_Neural_Networks_for_Medical_Image_Classification](https://www.researchgate.net/publication/394100604_Comparative_Analysis_of_Vision_Transformers_and_Convolutional_Neural_Networks_for_Medical_Image_Classification)
* [https://www.scitepress.org/Papers/2025/133819/133819.pdf](https://www.scitepress.org/Papers/2025/133819/133819.pdf)
* [https://www.researchgate.net/publication/394100604_Comparative_Analysis_of_Vision_Transformers_and_Convolutional_Neural_Networks_for_Medical_Image_Classification](https://www.researchgate.net/publication/394100604_Comparative_Analysis_of_Vision_Transformers_and_Convolutional_Neural_Networks_for_Medical_Image_Classification)
* [https://www.reddit.com/r/computervision/comments/1cu3pnw/cnn_vs_vision_transformer_a_practitioners_guide/](https://www.reddit.com/r/computervision/comments/1cu3pnw/cnn_vs_vision_transformer_a_practitioners_guide/)
* [https://www.atlantis-press.com/article/126011379.pdf](https://www.atlantis-press.com/article/126011379.pdf)
* [https://www.youtube.com/watch?v=2i3g52ZGsZI](https://www.youtube.com/watch?v=2i3g52ZGsZI)

In [76]:
# Self-Evaluation: Using a second pass of the LLM to grade the report based on a rubric
def self_evaluation(final_state, task_name):
    eval_prompt = f"""Evaluate this research report on '{task_name}' using this rubric:
    1. Coverage (1-10): Did it address key parts of the question?
    2. Faithfulness (1-10): Are claims supported by the provided sources?
    3. Hallucination Rate (1-10, lower is better): Are there unsupported claims?
    4. Usefulness (1-10): Is it decision-supportive?

    REPORT: {final_state}
    """
    return llm.invoke(eval_prompt).content

# Executing the evaluation for the generated report
print(self_evaluation(final_state['report'], "CNN vs ViT for Medical Imaging"))

Based on the provided research report, I'll evaluate it using the given rubric:

1. **Coverage (8/10)**: The report addresses key parts of the question, including the performance comparison, application areas, and computational efficiency of CNNs and ViTs for medical imaging. However, it could delve deeper into the technical implementation and provide more specific examples of medical imaging tasks where one architecture outperforms the other.

2. **Faithfulness (9/10)**: The report provides a comprehensive review of the trade-offs between CNNs and ViTs, and most claims are supported by the provided sources. The references include a mix of academic papers, blog posts, and online forums, which adds to the report's credibility. However, some sources, such as Reddit threads and YouTube videos, may not be considered traditional academic sources.

3. **Hallucination Rate (6/10)**: The report contains some unsupported claims, such as the statement that "ViTs have shown superior performance i

In [77]:
# Export: Converting the Markdown report into a styled HTML file and download it locally
from google.colab import files
from markdown import markdown

def export_to_html_report(report_markdown, filename="Research_Report.html"):
    style = """
    <style>
        body { font-family: sans-serif; line-height: 1.6; max-width: 800px; margin: auto; padding: 40px; color: #333; }
        h1 { color: #2c3e50; border-bottom: 2px solid #eee; }
        h2 { color: #34495e; margin-top: 30px; }
        pre { background: #f4f4f4; padding: 10px; border-radius: 5px; overflow-x: auto; }
        blockquote { border-left: 5px solid #eee; padding-left: 20px; font-style: italic; }
    </style>
    """

    html_content = style + markdown(report_markdown)

    with open(filename, "w") as f:
        f.write(html_content)

    print(f"HTML Report generated: {filename}")
    files.download(filename)

# Executing the export and download
export_to_html_report(final_state['report'], "Medical_Imaging_Research.html")

HTML Report generated: Medical_Imaging_Research.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>